# Augmented Multi-Expert NLI
This notebook combines **Synthetic Data Generation (T5)** with a **Mixture of Experts (MoE)** classifier. 

### Pipeline:
1. **Generator**: A T5 model trained to create synthetic hypotheses.
2. **Augmentation**: Expanding the training set with generated examples.
3. **MoE Classifier**: A specialized model with heads for Semantics, Entities, Actions, and Logic.

In [1]:
!pip install datasets transformers torch spacy pandas numpy scikit-learn
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 31.6 MB/s eta 0:00:0031m33.7 MB/s eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import spacy
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    T5Tokenizer, 
    T5ForConditionalGeneration, 
    TrainingArguments, 
    Trainer
)
from sklearn.metrics import f1_score, accuracy_score

nlp = spacy.load("en_core_web_sm", disable=["ner"])
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Synthetic Data Generation (T5)
We first train a T5 model to generate hypotheses. This helps the model generalize by providing varied linguistic structures for the same labels.

In [3]:
gen_model_name = "t5-small" # Using small for speed, t5-base preferred for quality
gen_tokenizer = T5Tokenizer.from_pretrained(gen_model_name)
generator = T5ForConditionalGeneration.from_pretrained(gen_model_name)

def preprocess_gen(example):
    input_text = f"generate hypothesis: premise: {example['premise']} label: {example['label']}"
    inputs = gen_tokenizer(input_text, truncation=True, padding="max_length", max_length=128)
    targets = gen_tokenizer(example["hypothesis"], truncation=True, padding="max_length", max_length=128)
    inputs["labels"] = targets["input_ids"]
    return inputs

# Load data
train_df = pd.read_csv("training_data/NLI/train.csv")
dev_df = pd.read_csv("training_data/NLI/dev.csv")
train_ds = Dataset.from_pandas(train_df)

gen_ds = train_ds.map(preprocess_gen)

gen_args = TrainingArguments(
    output_dir="t5_gen_results",
    per_device_train_batch_size=16,
    num_train_epochs=1, # 3 is ideal, 1 for demonstration
    report_to="none"
)

gen_trainer = Trainer(model=generator, args=gen_args, train_dataset=gen_ds)
gen_trainer.train()

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1527, training_loss=0.4048662985081601, metrics={'train_runtime': 79.4384, 'train_samples_per_second': 307.559, 'train_steps_per_second': 19.222, 'total_flos': 826667723390976.0, 'train_loss': 0.4048662985081601, 'epoch': 1.0})

In [4]:
def augment_data(dataset, n_samples=2000):
    generator.eval()
    synthetic_examples = []
    subset = dataset.select(range(min(n_samples, len(dataset))))
    
    for ex in subset:
        prompt = f"generate hypothesis: premise: {ex['premise']} label: {ex['label']}"
        inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)
        outputs = generator.generate(**inputs, max_length=64)
        gen_hyp = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        synthetic_examples.append({
            "premise": ex["premise"], 
            "hypothesis": gen_hyp, 
            "label": ex["label"]
        })
    return pd.concat([pd.DataFrame(dataset), pd.DataFrame(synthetic_examples)])

augmented_train_df = augment_data(train_ds)
print(f"Original size: {len(train_df)} | Augmented size: {len(augmented_train_df)}")

Original size: 24432 | Augmented size: 26432


## 2. MoE Model Architecture
We use the linguistic POS-tagging approach to create specialized expert inputs for the augmented data.

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def get_pos_filtered_text(text, pos_tags):
    doc = nlp(str(text))
    tokens = [token.text for token in doc if token.pos_ in pos_tags]
    return " ".join(tokens) if tokens else "none"

def extract_logic_features(premise, hypothesis):
    p_doc, h_doc = nlp(str(premise)), nlp(str(hypothesis))
    neg_p = sum(1 for t in p_doc if t.dep_ == "neg")
    neg_h = sum(1 for t in h_doc if t.dep_ == "neg")
    p_set = {t.lemma_.lower() for t in p_doc if not t.is_stop and not t.is_punct}
    h_set = {t.lemma_.lower() for t in h_doc if not t.is_stop and not t.is_punct}
    overlap = len(p_set & h_set) / len(p_set | h_set) if (p_set | h_set) else 0.0
    return [float(neg_p), float(neg_h), float(abs(neg_p - neg_h)), float(overlap)]

def preprocess_moe(example):
    main_enc = tokenizer(example["premise"], example["hypothesis"], truncation=True, padding="max_length", max_length=128)
    p_nouns = get_pos_filtered_text(example["premise"], ["NOUN", "PROPN"])
    h_nouns = get_pos_filtered_text(example["hypothesis"], ["NOUN", "PROPN"])
    p_verbs = get_pos_filtered_text(example["premise"], ["VERB"])
    h_verbs = get_pos_filtered_text(example["hypothesis"], ["VERB"])
    
    ent_enc = tokenizer(p_nouns, h_nouns, truncation=True, padding="max_length", max_length=128)
    act_enc = tokenizer(p_verbs, h_verbs, truncation=True, padding="max_length", max_length=128)
    
    return {
        "input_ids": main_enc["input_ids"],
        "attention_mask": main_enc["attention_mask"],
        "entity_ids": ent_enc["input_ids"],
        "action_ids": act_enc["input_ids"],
        "logic_features": extract_logic_features(example["premise"], example["hypothesis"]),
        "label": int(example["label"])
    }

In [6]:
class POSSpecializedMoE(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_labels=2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h_size = self.encoder.config.hidden_size

        self.semantic_expert = nn.Linear(h_size, num_labels)
        self.entity_expert = nn.Linear(h_size, num_labels)
        self.action_expert = nn.Linear(h_size, num_labels)
        self.logic_expert = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, num_labels))

        self.gating = nn.Sequential(nn.Linear(h_size + 4, 64), nn.ReLU(), nn.Linear(64, 4))
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, entity_ids, action_ids, logic_features, labels=None):
        sem_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        ent_out = self.encoder(input_ids=entity_ids).last_hidden_state[:, 0, :]
        act_out = self.encoder(input_ids=action_ids).last_hidden_state[:, 0, :]

        l_sem, l_ent = self.semantic_expert(sem_out), self.entity_expert(ent_out)
        l_act, l_log = self.action_expert(act_out), self.logic_expert(logic_features.float())

        gate_input = torch.cat([sem_out, logic_features.float()], dim=1)
        gate_weights = torch.softmax(self.gating(gate_input), dim=1)

        final_logits = (gate_weights[:, 0:1] * l_sem + gate_weights[:, 1:2] * l_ent + 
                        gate_weights[:, 2:3] * l_act + gate_weights[:, 3:4] * l_log)

        loss = self.loss_fn(final_logits, labels) if labels is not None else None
        return {"loss": loss, "logits": final_logits}

## 3. Training the Final MoE Model

In [7]:
final_train_ds = Dataset.from_pandas(augmented_train_df).map(preprocess_moe)
final_dev_ds = Dataset.from_pandas(dev_df).map(preprocess_moe)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1_score(labels, preds, average="macro")}

model = POSSpecializedMoE()
moe_args = TrainingArguments(
    output_dir="moe_final_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model, 
    args=moe_args, 
    train_dataset=final_train_ds, 
    eval_dataset=final_dev_ds, 
    compute_metrics=compute_metrics
)

trainer.train()
print(trainer.evaluate())

Map:   0%|          | 0/26432 [00:00<?, ? examples/s]

Map:   0%|          | 0/6736 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.473952,0.411963,0.816657,0.816234
2,0.318362,0.453522,0.827049,0.826849


KeyboardInterrupt: 